<a href="https://colab.research.google.com/github/arthisathish02042025-wq/Healthcare_Data_Cleaning/blob/main/Ecommerce_Sales_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Synthetic Sales Data created for accessing the data from e-commerce company

In [1]:
import pandas as pd
import random

random.seed(42)

# Define data parameters
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
salespersons = ['Alice', 'Bob', 'Carol', 'David', 'Emma', 'Frank']

# Generate 200 sales transactions
data = {
    'transaction_id': range(1001, 1201),
    'region': [random.choice(regions) for _ in range(200)],
    'category': [random.choice(categories) for _ in range(200)],
    'salesperson': [random.choice(salespersons) for _ in range(200)],
    'sales_amount': [round(random.uniform(50, 5000), 2) for _ in range(200)],
    'customer_id': [random.randint(5000, 5100) for _ in range(200)]
}

df = pd.DataFrame(data)
print(df.head(10))
print(f"\nDataset shape: {df.shape}")


   transaction_id region     category salesperson  sales_amount  customer_id
0            1001  North  Electronics        Emma       4000.80         5075
1            1002  North        Books       Alice       3634.70         5084
2            1003   East  Electronics       David       4210.50         5025
3            1004  South  Electronics        Emma       4601.73         5054
4            1005  South     Clothing         Bob       4904.57         5014
5            1006  South     Clothing       Carol       2693.91         5069
6            1007  North       Sports       Alice       4539.30         5028
7            1008  North       Sports       Frank       2979.87         5082
8            1009   West       Sports       David       3331.85         5019
9            1010  North     Clothing       Alice        465.54         5034

Dataset shape: (200, 6)


Task 1: Basic Grouping and Single Aggregations

In [3]:
#Calculate total sales amount for each region using groupby() and sum()
# 1. Total sales amount for each region, sorted descending
region_sales = df.groupby('region')['sales_amount'].sum().reset_index()
region_sales = region_sales.sort_values(by='sales_amount', ascending=False)

#Count the number of transactions for each product category using groupby() and count()
# 2. Transaction count for each product category
category_counts = df.groupby('category')['transaction_id'].count().reset_index()
category_counts.columns = ['category', 'transaction_count']

#Calculate the average sales amount per salesperson using groupby() and mean()
# 3. Average sales amount per salesperson
salesperson_avg = df.groupby('salesperson')['sales_amount'].mean().reset_index()
salesperson_avg.columns = ['salesperson', 'average_sales']

# Displaying Results
print("=== Top Performing Regions (Sorted) ===")
print(region_sales)

print("\n=== Transaction Counts by Category ===")
print(category_counts)

print("\n=== Average Sales per Salesperson ===")
print(salesperson_avg)

=== Top Performing Regions (Sorted) ===
  region  sales_amount
2  South     158977.36
1  North     135181.16
3   West     109383.07
0   East      95189.81

=== Transaction Counts by Category ===
        category  transaction_count
0          Books                 39
1       Clothing                 42
2    Electronics                 45
3  Home & Garden                 43
4         Sports                 31

=== Average Sales per Salesperson ===
  salesperson  average_sales
0       Alice    1998.432273
1         Bob    2554.919063
2       Carol    2454.368571
3       David    2743.036444
4        Emma    2493.457273
5       Frank    2464.135750


Task 2: Multi-Column Grouping and Multiple Aggregations

In [4]:
#Group by both region AND category to calculate total sales for each combination
# 1. Group by Region and Category for total sales
region_category_sales = df.groupby(['region', 'category'])['sales_amount'].sum().reset_index()

#For each salesperson, calculate three metrics simultaneously using the agg() method:
#Total sales ('sum')
#Average sales ('mean')
#Number of transactions ('count')
# 2. Multi-metric aggregation for each Salesperson
salesperson_metrics = df.groupby('salesperson')['sales_amount'].agg(['sum', 'mean', 'count']).reset_index()

#Sort the salesperson results by total sales in descending order to identify the top performer
# 3.Sort by total sales (sum) in descending order
salesperson_metrics = salesperson_metrics.sort_values(by='sum', ascending=False)

#Use .idxmax() on the grouped category sales to find which category has the maximum total revenue
# 4. Find the category with the absolute maximum total revenue
# First, group by category to get total revenue per category
category_totals = df.groupby('category')['sales_amount'].sum()
top_category_name = category_totals.idxmax()
top_category_value = category_totals.max()

# Display Results
print("=== Region and Category Sales (Partial View) ===")
print(region_category_sales.head(10))

print("\n=== Salesperson Performance Metrics (Sorted by Sum) ===")
print(salesperson_metrics)

print(f"\n=== Top Revenue Category ===\nCategory: {top_category_name} | Total: ${top_category_value:,.2f}")

=== Region and Category Sales (Partial View) ===
  region       category  sales_amount
0   East          Books      20027.53
1   East       Clothing      19926.20
2   East    Electronics      22791.55
3   East  Home & Garden      15949.08
4   East         Sports      16495.45
5  North          Books      42592.53
6  North       Clothing      18959.06
7  North    Electronics      38889.84
8  North  Home & Garden      19344.48
9  North         Sports      15395.25

=== Salesperson Performance Metrics (Sorted by Sum) ===
  salesperson        sum         mean  count
3       David  123436.64  2743.036444     45
5       Frank   98565.43  2464.135750     40
4        Emma   82284.09  2493.457273     33
1         Bob   81757.41  2554.919063     32
2       Carol   68722.32  2454.368571     28
0       Alice   43965.51  1998.432273     22

=== Top Revenue Category ===
Category: Electronics | Total: $114,464.29


Task 3: Custom Aggregation and Complete Sales Report

In [6]:
import pandas as pd

#Define a custom aggregation function that calculates the sales range (max - min) for each group
# 1. Define custom aggregation function for Sales Range
def sales_range(x):
    return x.max() - x.min()

#Apply this custom function along with standard aggregations to analyze sales by region
# 2. Apply custom and standard aggregations by Region
regional_range_report = df.groupby('region')['sales_amount'].agg(
    ['sum', 'mean', 'min', 'max', sales_range]
).reset_index()

#Create a final summary report that shows for each region
#Total number of transactions (using customer_id with count)
#Total sales amount
#Average transaction value
# 3. Create Final Summary Report using Dictionary Syntax
final_summary = df.groupby('region').agg({
    'sales_amount': ['sum', 'mean'],
    'customer_id': 'count'
}).reset_index()

# Display Results
print("=== Regional Sales Analysis with Custom Range ===")
print(regional_range_report)

print("\n=== Final Summary Sales Report ===")
print(final_summary)

=== Regional Sales Analysis with Custom Range ===
  region        sum         mean     min      max  sales_range
0   East   95189.81  2213.716512  307.82  4758.46      4450.64
1  North  135181.16  2413.949286  110.71  4788.56      4677.85
2  South  158977.36  2789.076491  373.45  4983.33      4609.88
3   West  109383.07  2485.978864  203.60  4903.53      4699.93

=== Final Summary Sales Report ===
  region sales_amount              customer_id
                  sum         mean       count
0   East     95189.81  2213.716512          43
1  North    135181.16  2413.949286          56
2  South    158977.36  2789.076491          57
3   West    109383.07  2485.978864          44


**Explain the multi-level column structure that results from the dictionary-based aggregation and how it differs from single aggregations:**

Understanding the Multi-Level Column Structure
When we use the dictionary syntax in .agg(), Pandas creates a MultiIndex for the columns.

**Structure:** The top level is the original column name (e.g., sales_amount), and the second level is the aggregation function name (e.g., sum).

**How it differs:** In single aggregations, we get a "flat" DataFrame where each column is just a single string. In dictionary-based aggregation, the columns are tuples like ('sales_amount', 'sum').

**Accessing Data:** To flatten this for easier reporting, we can rename the columns using final_summary.columns = ['region', 'total_sales', 'avg_sales', 'transaction_count'].